# 04 — Feature Engineering & Market State Representation
**Project:** Reinforcement Learning with RNN-Based Market State Representation for Algorithmic Trading  
**Step:** Compute all technical indicators and stationary market-state features from raw OHLCV data.

**Engineered features (14 state inputs + 1 target):**
| Feature | Category |
|---|---|
| `Daily_Return`, `Log_Return` | Return metrics |
| `HL_Pct`, `Rolling_Vol_20` | Volatility |
| `SMA_Ratio_10/20/50` | Trend (stationary) |
| `EMA_Ratio_12/26` | Trend (stationary) |
| `RSI_14_Norm` | Momentum oscillator |
| `MACD_Line/Signal/Hist_Norm` | Momentum |
| `Volume_Ratio_20` | Liquidity |
| `Target_Return_Next` | RL target (next-day return) |

## 1. Imports & Paths

In [ ]:
import os
import sys
import logging
from typing import Dict, List, Optional

import numpy as np
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

RAW_DATA_DIR   = os.path.join("data", "raw")
PROCESSED_DIR  = os.path.join("data", "processed")
REPORT_DIR     = os.path.join("reports", "data_preparation")
DOC_PATH       = os.path.join(REPORT_DIR, "feature_documentation.md")

STOCK_SYMBOLS  = ["RELIANCE", "TCS", "INFY", "HDFCBANK", "ITC"]
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

## 2. RSI Calculation Function

In [ ]:
def calculate_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    """
    Calculates Wilder's Relative Strength Index (RSI-14).

    Formula:
        delta = Close_t - Close_{t-1}
        gain  = max(delta, 0),  loss = max(-delta, 0)
        smoothed via Wilder EMA (alpha = 1/period)
        RS    = smoothed_gain / smoothed_loss
        RSI   = 100 - 100 / (1 + RS)
    """
    delta = series.diff()
    gain  = delta.clip(lower=0.0)
    loss  = (-delta).clip(lower=0.0)

    avg_gain = gain.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()

    rs  = avg_gain / avg_loss.replace(0.0, np.nan)
    rsi = 100.0 - (100.0 / (1.0 + rs))
    return rsi.fillna(50.0)

## 3. Feature Engineering Function

In [ ]:
def compute_market_features(df: pd.DataFrame, symbol: str) -> pd.DataFrame:
    """
    Computes all stationary market-state features for a single asset.

    Look-ahead bias prevention: every feature at bar t depends strictly on
    data from bar t and earlier. Only Target_Return_Next uses bar t+1 close.

    Args:
        df:     Raw OHLCV DataFrame.
        symbol: Asset ticker symbol.

    Returns:
        Processed DataFrame with 33 columns (raw OHLCV + 28 derived + target).
    """
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    df["Symbol"] = symbol

    close  = df["Close"]
    high   = df["High"]
    low    = df["Low"]
    volume = df["Volume"]

    # --- Return Metrics ---
    df["Daily_Return"] = close.pct_change(fill_method=None)
    df["Log_Return"]   = np.log(close / close.shift(1))

    # --- Volatility ---
    df["HL_Pct"]          = (high - low) / close
    df["Rolling_Vol_20"]  = df["Daily_Return"].rolling(window=20).std()

    # --- Simple Moving Averages (10, 20, 50) ---
    df["SMA_10"] = close.rolling(window=10).mean()
    df["SMA_20"] = close.rolling(window=20).mean()
    df["SMA_50"] = close.rolling(window=50).mean()

    # Stationary price-to-SMA ratios
    df["SMA_Ratio_10"] = (close - df["SMA_10"]) / df["SMA_10"]
    df["SMA_Ratio_20"] = (close - df["SMA_20"]) / df["SMA_20"]
    df["SMA_Ratio_50"] = (close - df["SMA_50"]) / df["SMA_50"]

    # --- Exponential Moving Averages (12, 26) ---
    df["EMA_12"] = close.ewm(span=12, adjust=False).mean()
    df["EMA_26"] = close.ewm(span=26, adjust=False).mean()

    df["EMA_Ratio_12"] = (close - df["EMA_12"]) / df["EMA_12"]
    df["EMA_Ratio_26"] = (close - df["EMA_26"]) / df["EMA_26"]

    # --- Momentum: RSI-14 ---
    df["RSI_14"]      = calculate_rsi(close, period=14)
    df["RSI_14_Norm"] = df["RSI_14"] / 100.0

    # --- Momentum: MACD (12, 26, 9) ---
    df["MACD_Line"]   = df["EMA_12"] - df["EMA_26"]
    df["MACD_Signal"] = df["MACD_Line"].ewm(span=9, adjust=False).mean()
    df["MACD_Hist"]   = df["MACD_Line"] - df["MACD_Signal"]

    # Normalised MACD (stationary: divided by Close)
    df["MACD_Line_Norm"]   = df["MACD_Line"]   / close
    df["MACD_Signal_Norm"] = df["MACD_Signal"] / close
    df["MACD_Hist_Norm"]   = df["MACD_Hist"]   / close

    # --- Liquidity: Volume Ratio ---
    df["Volume_SMA_20"]   = volume.rolling(window=20).mean()
    df["Volume_Ratio_20"] = volume / df["Volume_SMA_20"].replace(0.0, np.nan)
    df["Volume_Ratio_20"] = df["Volume_Ratio_20"].fillna(1.0)

    # --- Target: Next-Day Return (EXCLUDED from state features) ---
    df["Target_Return_Next"] = close.shift(-1).sub(close).div(close)
    df["Target_Date_Next"]   = df["Date"].shift(-1).dt.strftime("%Y-%m-%d")

    # Format Date back to string
    df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")

    # Drop 50-bar warm-up and final row (no next-day target)
    burn_in   = 50
    raw_len   = len(df)
    df_clean  = df.iloc[burn_in:-1].copy().reset_index(drop=True)

    logger.info(
        f"[{symbol}] {raw_len} raw → {len(df_clean)} stationary bars "
        f"(dropped {burn_in} warmup + 1 trailing target bar)"
    )
    return df_clean

## 4. Run Feature Engineering for All Stocks

In [ ]:
print("=" * 60)
print("STEP 3: Feature Engineering (Market State Representation)")
print("=" * 60)

processed_dfs: Dict[str, pd.DataFrame] = {}

for symbol in STOCK_SYMBOLS:
    raw_path = os.path.join(RAW_DATA_DIR, f"{symbol}.csv")
    if not os.path.exists(raw_path):
        raise FileNotFoundError(f"Raw file missing: {raw_path}")

    raw_df  = pd.read_csv(raw_path)
    proc_df = compute_market_features(raw_df, symbol)

    out_path = os.path.join(PROCESSED_DIR, f"{symbol}_processed.csv")
    proc_df.to_csv(out_path, index=False)
    processed_dfs[symbol] = proc_df

    logger.info(f"Saved {symbol}_processed.csv  ({proc_df.shape[0]} rows × {proc_df.shape[1]} cols)")

print("\nFeature engineering complete.")

## 5. Processed Dataset Summary

In [ ]:
STATE_FEATURE_COLS = [
    "Daily_Return", "Log_Return", "HL_Pct", "Rolling_Vol_20",
    "SMA_Ratio_10", "SMA_Ratio_20", "SMA_Ratio_50",
    "EMA_Ratio_12", "EMA_Ratio_26", "RSI_14_Norm",
    "MACD_Line_Norm", "MACD_Signal_Norm", "MACD_Hist_Norm", "Volume_Ratio_20",
]

print(f"{'Symbol':<10} {'Rows':>6}  {'Cols':>5}  {'Start Date':>12}  {'End Date':>12}  {'NaN count':>10}")
print("-" * 65)
for sym, df in processed_dfs.items():
    print(
        f"{sym:<10} {len(df):>6}  {len(df.columns):>5}  "
        f"{df['Date'].min():>12}  {df['Date'].max():>12}  "
        f"{df[STATE_FEATURE_COLS].isna().sum().sum():>10}"
    )

## 6. Save Feature Documentation

In [ ]:
\
doc = r'''# Feature Engineering & Market State Representation Documentation

**Project:** Reinforcement Learning with RNN-Based Market State Representation for Algorithmic Trading  
**Author:** NeuroTrade Research Pipeline  
**Version:** 1.0.0  

---

## 1. Mathematical Specifications of Engineered Features

The state representation vector $s_t$ must satisfy two fundamental machine learning properties:
1. **Stationarity:** The statistical properties (mean, variance) of the input features should remain stable across different historical price regimes.
2. **Causality (No Look-Ahead Bias):** Every indicator at time $t$ depends strictly on information set $\mathcal{F}_t = \{O_i, H_i, L_i, C_i, V_i\}_{i=0}^t$.

| Feature Name | Category | Mathematical Definition | Financial Rationale |
|---|---|---|---|
| `Daily_Return` | Return Metric | $R_t = (C_t - C_{t-1}) / C_{t-1}$ | Standard daily arithmetic price return. |
| `Log_Return` | Return Metric | $r_t = \ln(C_t / C_{t-1})$ | Continuously compounded return. |
| `HL_Pct` | Volatility | $(H_t - L_t) / C_t$ | Intraday price dispersion normalised by close. |
| `Rolling_Vol_20` | Volatility | 20-day rolling std of Daily_Return | Realised return volatility. |
| `SMA_Ratio_10/20/50` | Trend | $(C_t - SMA_{n,t}) / SMA_{n,t}$ | Percentage distance from moving average. |
| `EMA_Ratio_12/26` | Trend | $(C_t - EMA_{n,t}) / EMA_{n,t}$ | Scale-invariant fast/slow trend deviation. |
| `RSI_14_Norm` | Momentum | $RSI_{14} / 100 \in [0,1]$ | Wilder smoothed momentum oscillator. |
| `MACD_Line/Signal/Hist_Norm` | Momentum | MACD components divided by Close | Stationary momentum convergence metrics. |
| `Volume_Ratio_20` | Liquidity | $V_t / SMA_{20}(V)$ | Relative volume; identifies conviction moves. |
| `Target_Return_Next` | RL Target | $(C_{t+1} - C_t) / C_t$ | Next-day return — excluded from state features. |

---

## 2. Stationary State Feature Subset for RNN

```python
STATE_FEATURE_COLS = [
    "Daily_Return", "Log_Return", "HL_Pct", "Rolling_Vol_20",
    "SMA_Ratio_10", "SMA_Ratio_20", "SMA_Ratio_50",
    "EMA_Ratio_12", "EMA_Ratio_26", "RSI_14_Norm",
    "MACD_Line_Norm", "MACD_Signal_Norm", "MACD_Hist_Norm", "Volume_Ratio_20",
]
```

---

## 3. Warmup Burn-in & Integrity Guarantees

- **50-Day Burn-in:** SMA-50 requires 50 bars; initial 50 rows are cleanly dropped.
- **Trailing Row Removal:** Final row (where next-day close is unavailable) is dropped.
- **No Look-Ahead Bias:** Every feature at row $t$ is independent of rows $t+1, t+2, \ldots$
'''

os.makedirs(REPORT_DIR, exist_ok=True)
with open(DOC_PATH, "w", encoding="utf-8") as f:
    f.write(doc)
print(f"Feature documentation saved → {DOC_PATH}")